In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))
from hough_sam.constants import FRAMES_DIR, SEG_RESULTS_DIR, FIGURES_DIR, CHECKPOINTS_DIR, MODELS_DIR, LABELS_FILE, SEG_VIDEOS_DIR

import os
import pandas as pd
from pathlib import Path

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load Data

In [ ]:
def load_pickled_dataframes_from_directory(base_directory) -> dict:
    """
    Iterates through all folders in a given directory, loads pickled pandas DataFrames
    (files ending with '.pkl') into a list, and returns the list.

    Args:
        base_directory (str): The root directory to start searching from.

    Returns:
        dict: A dict of loaded pandas DataFrames.
    """
    results = {}
    p = Path(base_directory)
    for item in p.iterdir():
        if item.is_dir():
            file = f"{item.name}/{item.name}.pkl"
            file_path = os.path.join(base_directory, file)
            try:
                df = pd.read_pickle(file_path)
                if isinstance(df, pd.DataFrame):
                    case_index = int(item.name[5:9])
                    results[case_index] = df
                    print(f"Loaded DataFrame from: {file_path}")
                else:
                    print(f"File {file_path} is not a pandas DataFrame, skipping.")
            except Exception as e:
                print(f"Error loading {file_path}: {e}")
    return dict(results)

In [ ]:
dir = SEG_RESULTS_DIR

# Now, use the function to load them
processed_cases = load_pickled_dataframes_from_directory(dir)

Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2000_50fps/case_2000_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2001_50fps/case_2001_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2002_50fps/case_2002_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2003_50fps/case_2003_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2004_50fps/case_2004_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2005_50fps/case_2005_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2006_50fps/case_2006_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2007_50fps/case_2007_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2008_50fps/case_2008_50fps.pkl
Loaded DataFrame from: /content/drive/MyDrive/Cataract/seg_results/case_2

In [ ]:
processed_cases.keys()

dict_keys([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2048, 2049, 5013, 5014, 5015, 5016, 5017, 5032, 5051, 5057, 5058, 5072, 5104, 5180, 5299, 5300, 5301, 5303, 5304, 5305, 5309, 5315, 5316, 5317, 5319, 5325, 5329, 5334, 5335, 5340, 5353])

# Smoothing

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Iterable, Optional, Tuple
import numpy as np
import pandas as pd


def _impute_cell(a: pd.Series, b: pd.Series, f: int) -> pd.Series:

  result = a.copy()
  result["frame_index"] = f
  if a["trapezoid_coord"] is not None and b["trapezoid_coord"] is not None:
    result["trapezoid_coord"] = [[(c1[0]+c2[0])/2, (c1[1]+c2[1])/2] for c1, c2 in zip(a["trapezoid_coord"], b["trapezoid_coord"])]

  result["trapezoid_area"] = (a["trapezoid_area"] + b["trapezoid_area"]) / 2

  if a["trapezoid_s1"] is not None and b["trapezoid_s1"] is not None:
    result["trapezoid_s1"] = [[(c1[0]+c2[0])/2, (c1[1]+c2[1])/2] for c1, c2 in zip(a["trapezoid_s1"], b["trapezoid_s1"])]

  if a["trapezoid_s2"] is not None and b["trapezoid_s2"] is not None:
    result["trapezoid_s2"] = [[(c1[0]+c2[0])/2, (c1[1]+c2[1])/2] for c1, c2 in zip(a["trapezoid_s2"], b["trapezoid_s2"])]

  result["side_ratio"] = (a["side_ratio"] + b["side_ratio"]) / 2

  return result

def find_longest_continuous_detection( # check if this harm the data by choosing noise
    df: pd.DataFrame,
    max_tolerable_jump: int = 2
) -> Tuple[int, int]:
    """
    Find [start_frame, stop_frame] (inclusive) of the longest continuous detection segment.

    A "continuous" segment allows up to `max_tolerable_jump` consecutive *missing* frames
    between two detected frames. (So a gap of length 1..max_tolerable_jump is tolerated.)

    Returns:
      (start_frame, stop_frame) inclusive, as ints.
    """
    if len(df) == 0:
        raise ValueError("df is empty.")

    df = df.copy()
    try:
        df.index = df.index.astype(int)
    except Exception as e:
        raise ValueError("df.index must be integer frame indices (or convertible to int).") from e

    det_mask = df["trapezoid_coord"].isnull() == False
    det_frames = df.index[det_mask.values].to_numpy(dtype=int)
    if len(det_frames) == 0:
        raise ValueError("No detected frames found; cannot define a continuous detection segment.")

    # Build segments by tolerated gaps between successive detections
    segments: list[Tuple[int, int]] = []
    seg_start = int(det_frames[0])
    prev = int(det_frames[0])

    for f in det_frames[1:]:
        f = int(f)
        missing_count = f - prev - 1
        if missing_count <= max_tolerable_jump:
            prev = f
        else:
            segments.append((seg_start, prev))
            seg_start = f
            prev = f
    segments.append((seg_start, prev))

    # Choose longest by span (inclusive length)
    best_start, best_stop = max(segments, key=lambda s: (s[1] - s[0] + 1, -s[0]))
    return int(best_start), int(best_stop)


def impute_gaps_within_segment(
    df: pd.DataFrame,
    start_frame: int,
    stop_frame: int,
    max_tolerable_jump: int = 2
) -> pd.DataFrame:
    """
    Impute missing frames inside [start_frame, stop_frame] (inclusive) for *tolerated gaps* only.

    For any gap between two detected frames of length <= max_tolerable_jump, impute each missing
    frame in the gap with the per-column average of the values at the two bounding detected frames.

    Only imputes columns in value_cols. Frames outside the segment are untouched.
    """
    if start_frame > stop_frame:
        raise ValueError("start_frame must be <= stop_frame")
    if len(df) == 0:
        return df

    out = df.copy()
    try:
        out.index = out.index.astype(int)
    except Exception as e:
        raise ValueError("df.index must be integer frame indices (or convertible to int).") from e

    seg_mask = (out.index >= int(start_frame)) & (out.index <= int(stop_frame))
    det_mask = df["trapezoid_coord"].isnull() == False
    det_in_seg = det_mask.values & seg_mask
    det_frames_in_seg = out.index[det_in_seg].to_numpy(dtype=int)

    if len(det_frames_in_seg) < 2:
        return out  # nothing to impute "between two detections"

    # Impute within tolerated gaps
    for left, right in zip(det_frames_in_seg[:-1], det_frames_in_seg[1:]):
        gap = right - left - 1
        if gap <= 0 or gap > max_tolerable_jump:
            continue

        for missing_frame in range(left + 1, right):
            # impute
            row_left = out.iloc[left]
            row_right = out.iloc[right]
            out.iloc[missing_frame] = _impute_cell(row_left, row_right, missing_frame)

    return out



def smooth(
    df: pd.DataFrame,
    max_tolerable_jump: int = 2,
) -> Tuple[int, int, pd.DataFrame]:
    """Helper that composes the two steps."""
    s, e = find_longest_continuous_detection(
        df,
        max_tolerable_jump=max_tolerable_jump
    )
    out = impute_gaps_within_segment(
        df,
        s,
        e,
        max_tolerable_jump=max_tolerable_jump,
    )
    return s, e, out


# Case-level Features

In [ ]:
from numpy._core.fromnumeric import mean
def calculate_case_level_features(case_df) -> dict:
  start, stop, case_df = smooth(case_df, max_tolerable_jump = 5) # increased this
  if start < stop:
    input = case_df.iloc[start:stop+1]
    geometric_features = calculate_case_level_geometric_feature(input, start, stop)
    area_features = calculate_case_level_area_feature(input, start, stop)
    detection_stability = calculate_case_level_detection_stability(start, stop)
    return {**geometric_features, **area_features, **detection_stability}
  else:
    return {}

def calculate_case_level_geometric_feature(features_by_frame, start, stop) -> dict:
  ratios = features_by_frame["side_ratio"]
  min_ratio = ratios.min()
  max_ratio = ratios.max()
  deviations = (ratios - 1).abs()
  mean_deviation = deviations.mean()
  max_deviation = deviations.max()
  return {"min_ratio": min_ratio,
          "max_ratio": max_ratio,
          "mean_deviation": mean_deviation.item(),
          "max_deviation": max_deviation}


def calculate_slopes(values: list) -> list:
    n = len(values)
    if n < 2:
        return []
    return [values[i-1] - values[i+1] for i in range(1, n-1, 2)]

def calculate_case_level_area_feature(features_by_frame, start, stop) -> dict:
  areas = features_by_frame["trapezoid_area"]
  min_area = areas.min()
  max_area = areas.max()
  max_index = areas.idxmax()
  slopes = calculate_slopes(areas.to_list())
  slopes_early = slopes[0:max_index-start]
  slopes_late = slopes[max_index-start:]
  return {"min_area": min_area,
          "max_area": max_area,
          "area_growth_slope_early": mean(slopes_early) if len(slopes_early) > 0 else 0, #early, mid, and late stage avg slope, and the number of frames within each stage
          "area_growth_slope_late": mean(slopes_late) if len(slopes_late) > 0 else 0} # choose 2 good cases and visualize the slopes per frame first


def calculate_case_level_detection_stability(start, stop) -> dict:
  return {"longest_detection_run": stop-start+1} # remove start, stop

# Process Case-level Features and Labels

In [ ]:
# calculate case level features
case_level_features_and_labels = {}
for case_index, case_df in processed_cases.items():
  try:
    case_level_features_and_labels[case_index] = calculate_case_level_features(case_df)
  except Exception as e:
    print(f"Error calculating case level features for case {case_index}: {e}")

Error calculating case level features for case 2015: No detected frames found; cannot define a continuous detection segment.
Error calculating case level features for case 2039: No detected frames found; cannot define a continuous detection segment.
Error calculating case level features for case 2045: No detected frames found; cannot define a continuous detection segment.
Error calculating case level features for case 5299: No detected frames found; cannot define a continuous detection segment.
Error calculating case level features for case 5329: No detected frames found; cannot define a continuous detection segment.


In [ ]:
case_level_features_and_labels[2003]

{'min_ratio': 0.6633751392364502,
 'max_ratio': 3.2962682247161865,
 'mean_deviation': 0.8569070100784302,
 'max_deviation': 2.2962682247161865,
 'min_area': 171.0,
 'max_area': 468.0,
 'area_growth_slope_early': np.float64(-21.125),
 'area_growth_slope_late': 0,
 'longest_detection_run': 26}

In [ ]:
# load labels
file = LABELS_FILE
labels = pd.read_excel(file)

In [ ]:
for index, row in labels.iterrows():
  case_name = row["case_name"]
  case_index = int(case_name[5:9])
  if case_index in case_level_features_and_labels:
    is_correct = row["Was the incision done correctly? (Y/N)"]
    incision_architecture_rating = row["Incision Architecture Rating (-1 / 0 / 1)"]
    incision_location_rating = row["Incision Location Rating (-1 / 0 / 1)"]
    incision_size = row["Incision Size (-1 / 0 / 1)"]
    case_level_features_and_labels[case_index] |= {
        "is_correct": is_correct == 'y',
        "incision_architecture_rating": incision_architecture_rating,
        "incision_location_rating": incision_location_rating,
        "incision_size": incision_size
    }


In [ ]:
case_level_features_and_labels[2003]

{'min_ratio': 0.6633751392364502,
 'max_ratio': 3.2962682247161865,
 'mean_deviation': 0.8569070100784302,
 'max_deviation': 2.2962682247161865,
 'min_area': 171.0,
 'max_area': 468.0,
 'area_growth_slope_early': np.float64(-21.125),
 'area_growth_slope_late': 0,
 'longest_detection_run': 26,
 'is_correct': True,
 'incision_architecture_rating': 1.0,
 'incision_location_rating': 1.0,
 'incision_size': 1.0}

In [ ]:
df = pd.DataFrame.from_dict(case_level_features_and_labels, orient="index")
df.to_csv(PROJECT_ROOT / "case_level_features_with_labels.csv", index=False)

# Label Prediction

## Data Augmentation

In [ ]:
from __future__ import annotations

from typing import Any, Dict, Hashable, List, Tuple
import numpy as np
import math
from sklearn.model_selection import train_test_split

In [ ]:
def stratified_split_case_indices(
    case_level_features: Dict[Hashable, Dict[str, Any]],
    label_col: str,
    train_ratio: float = 0.50,
    val_ratio: float = 0.25,
    test_ratio: float = 0.25,
    seed: int = 42,
) -> Tuple[List[Hashable], List[Hashable], List[Hashable]]:
    """
    Stratified train/val/test split by label_col.
    Assumes label_col exists for every case and is categorical (e.g., -1/0/1 or 0/1).
    """
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("train_ratio + val_ratio + test_ratio must sum to 1.0")

    case_ids = np.array(list(case_level_features.keys()), dtype=object)
    y = np.array([case_level_features[case_id][label_col] for case_id in case_ids], dtype=object)

    # First split: train vs (val+test)
    test_size_1 = val_ratio + test_ratio
    train_ids, temp_ids, y_train, y_temp = train_test_split(
        case_ids,
        y,
        test_size=test_size_1,
        random_state=seed,
        stratify=y,
    )

    # Second split: val vs test from the temp pool
    # val fraction within temp:
    val_frac_of_temp = val_ratio / (val_ratio + test_ratio)
    try:
      val_ids, test_ids = train_test_split(
          temp_ids,
          test_size=(1.0 - val_frac_of_temp),
          random_state=seed,
          stratify=y_temp
      )
    except:
      val_ids, test_ids = train_test_split(
          temp_ids,
          test_size=(1.0 - val_frac_of_temp),
          random_state=seed
      )

    return list(train_ids), list(val_ids), list(test_ids)


def label_distribution(case_level_features: Dict[Hashable, Dict[str, Any]], ids: List[Hashable], label_col: str):
    vals = [case_level_features[cid][label_col] for cid in ids]
    unique, counts = np.unique(vals, return_counts=True)
    return dict(zip(unique.tolist(), counts.tolist()))


In [ ]:
# Example:
to_del = []
for case_id in case_level_features_and_labels:
  if math.isnan(case_level_features_and_labels[case_id]["incision_architecture_rating"]):
    to_del.append(case_id)
for id in to_del:
  del case_level_features_and_labels[id]


train_case_ids_arc, val_case_ids_arc, test_case_ids_arc = stratified_split_case_indices(
    case_level_features_and_labels, label_col="incision_architecture_rating",
    train_ratio=0.60, val_ratio=0.20, test_ratio=0.20, seed=10
)

train_case_ids_loc, val_case_ids_loc, test_case_ids_loc = stratified_split_case_indices(
    case_level_features_and_labels, label_col="incision_location_rating",
    train_ratio=0.50, val_ratio=0.25, test_ratio=0.25, seed=10
)

train_case_ids_size, val_case_ids_size, test_case_ids_size = stratified_split_case_indices(
    case_level_features_and_labels, label_col="incision_size",
    train_ratio=0.60, val_ratio=0.20, test_ratio=0.20, seed=10
)

print("train_arc:", label_distribution(case_level_features_and_labels, train_case_ids_arc, "incision_architecture_rating"))
print("val_arc  :", label_distribution(case_level_features_and_labels, val_case_ids_arc, "incision_architecture_rating"))
print("test_arc :", label_distribution(case_level_features_and_labels, test_case_ids_arc, "incision_architecture_rating"))

print("train_loc:", label_distribution(case_level_features_and_labels, train_case_ids_loc, "incision_location_rating"))
print("val_loc  :", label_distribution(case_level_features_and_labels, val_case_ids_loc, "incision_location_rating"))
print("test_loc :", label_distribution(case_level_features_and_labels, test_case_ids_loc, "incision_location_rating"))

print("train_size:", label_distribution(case_level_features_and_labels, train_case_ids_size, "incision_size"))
print("val_size  :", label_distribution(case_level_features_and_labels, val_case_ids_size, "incision_size"))
print("test_size :", label_distribution(case_level_features_and_labels, test_case_ids_size, "incision_size"))

train_arc: {-1.0: 1, 0.0: 7, 1.0: 35}
val_arc  : {-1.0: 1, 0.0: 1, 1.0: 13}
test_arc : {0.0: 4, 1.0: 11}
train_loc: {-1.0: 2, 0.0: 7, 1.0: 27}
val_loc  : {-1.0: 1, 0.0: 3, 1.0: 14}
test_loc : {-1.0: 1, 0.0: 4, 1.0: 14}
train_size: {-1.0: 1, 0.0: 7, 1.0: 35}
val_size  : {-1.0: 1, 0.0: 1, 1.0: 13}
test_size : {0.0: 4, 1.0: 11}


In [ ]:
def augment(cases: dict, label_col: str = "incision_architecture_rating",
            aug_ratios: dict = {-1.0: 20, 0.0: 10, 1.0: 2},
            max_ratio: int = 20):

  def mask_10pct_random_frames(
      df: pd.DataFrame,
      frac: float = 0.10,
      seed = 42
  ) -> pd.DataFrame:
      """
      Mask ~10% frames randomly (among detected frames only).
      """
      rand = np.random.default_rng(seed)
      eligible = np.arange(start, stop+1)
      if eligible.size == 0:
          return df.copy(deep=True)

      k = max(1, int(round(eligible.size * frac)))
      chosen = rand.choice(eligible, size=min(k, eligible.size), replace=False)

      return df.drop(chosen), (eligible.size - k)

  results = {}
  i = 0
  for train_case_id in cases:
    case_df = processed_cases[train_case_id]
    start, stop, case_df = smooth(case_df, max_tolerable_jump = 5)
    if start < stop:
      input = case_df.iloc[start:stop+1]
      # augment one case specific times with differenct random seeds
      for j in range(aug_ratios[cases[train_case_id][label_col]]):
        augmented_df, size = mask_10pct_random_frames(input, seed = j+1)
        try:
          geometric_features = calculate_case_level_geometric_feature(augmented_df, start, stop)
          area_features = calculate_case_level_area_feature(augmented_df, start, stop)
          detection_stability = calculate_case_level_detection_stability(start, stop)
          detection_stability["longest_detection_run"] = size
          labels = ['is_correct', 'incision_architecture_rating', 'incision_location_rating', 'incision_size']
          ratings = {label: cases[train_case_id][label] for label in labels}
          results[i*max_ratio+j] = {**geometric_features, **area_features, **detection_stability, **ratings}
        except Exception as e:
          print(f"Error calculating case level features for case {case_index}: {e}")
      i = i + 1
  return results

In [ ]:
train_arc = {id: case_level_features_and_labels[id] for id in train_case_ids_arc if id in case_level_features_and_labels}
valid_arc = {id: case_level_features_and_labels[id] for id in val_case_ids_arc if id in case_level_features_and_labels}
test_arc = {id: case_level_features_and_labels[id] for id in test_case_ids_arc if id in case_level_features_and_labels}
augmented_arc = augment(train_arc, label_col="incision_architecture_rating")

train_loc = {id: case_level_features_and_labels[id] for id in train_case_ids_loc if id in case_level_features_and_labels}
valid_loc = {id: case_level_features_and_labels[id] for id in val_case_ids_loc if id in case_level_features_and_labels}
test_loc = {id: case_level_features_and_labels[id] for id in test_case_ids_loc if id in case_level_features_and_labels}
augmented_loc = augment(train_loc, label_col="incision_location_rating", aug_ratios = {-1.0: 20, 0.0: 8, 1.0: 2})

train_size = {id: case_level_features_and_labels[id] for id in train_case_ids_size if id in case_level_features_and_labels}
valid_size = {id: case_level_features_and_labels[id] for id in val_case_ids_size if id in case_level_features_and_labels}
test_size = {id: case_level_features_and_labels[id] for id in test_case_ids_size if id in case_level_features_and_labels}
augmented_size = augment(train_size, label_col="incision_size")

In [ ]:
print("augmented_arc:", label_distribution(augmented_arc, augmented_arc.keys(), "incision_architecture_rating"))
print("augmented_loc:", label_distribution(augmented_loc, augmented_loc.keys(), "incision_location_rating"))
print("augmented_size:", label_distribution(augmented_size, augmented_size.keys(), "incision_size"))

augmented_arc: {-1.0: 20, 0.0: 70, 1.0: 68}
augmented_loc: {-1.0: 40, 0.0: 56, 1.0: 54}
augmented_size: {-1.0: 20, 0.0: 70, 1.0: 70}


In [ ]:
train_augmented_arc = {**train_arc, **augmented_arc}
train_augmented_arc_df = pd.DataFrame.from_dict(train_augmented_arc, orient="index").dropna()
print(f"Size of augmented arc train set: {train_augmented_arc_df.shape[0]}")

train_augmented_loc = {**train_loc, **augmented_loc}
train_augmented_loc_df = pd.DataFrame.from_dict(train_augmented_loc, orient="index").dropna()
print(f"Size of augmented loc train set: {train_augmented_loc_df.shape[0]}")

train_augmented_size = {**train_size, **augmented_size}
train_augmented_size_df = pd.DataFrame.from_dict(train_augmented_size, orient="index").dropna()
print(f"Size of augmented size train set: {train_augmented_size_df.shape[0]}")

Size of augmented arc train set: 200
Size of augmented loc train set: 186
Size of augmented size train set: 171


In [ ]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier


def train_select_dt_and_test(
    train_augmented_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    label_col: str,
    *,
    # keep it simple: accuracy is the selection metric (you can change to "balanced_accuracy")
    selection_metric: str = "accuracy",
    # small grid, fast + sane
    grid: Optional[Dict[str, Any]] = None,
    random_state: int = 42,
    return_leaderboard: bool = True,
) -> Tuple[Pipeline, Dict[str, Any], Optional[pd.DataFrame]]:
    """
    Train multiple DecisionTreeClassifier models on train_augmented_df, pick best on valid_df,
    then report performance on test_df.

    Assumptions:
      - Each df contains the label_col
      - All other columns are numeric features (or coercible to numeric)
      - Missing values may exist (None/NaN); median imputation is used
      - Indices can be case_ids; preserved but not required

    Returns:
      best_model: fitted sklearn Pipeline (imputer + decision tree)
      report: dict with metrics + confusion matrices + text report
      leaderboard: optional DataFrame of hyperparams and valid score
    """
    if label_col not in train_augmented_df.columns:
        raise ValueError(f"{label_col=} not in train_augmented_df columns")
    if label_col not in valid_df.columns:
        raise ValueError(f"{label_col=} not in valid_df columns")
    if label_col not in test_df.columns:
        raise ValueError(f"{label_col=} not in test_df columns")

    def _Xy(df: pd.DataFrame):
        X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
        y = df[label_col].to_numpy()
        return X, y

    try:
      X_train, y_train = _Xy(train_augmented_df)
      X_val, y_val = _Xy(valid_df)
      X_test, y_test = _Xy(test_df)
    except Exception as e:
      print(f"Error converting to numeric: {e}")
      return

    metric_fn = {
        "accuracy": accuracy_score,
        "balanced_accuracy": balanced_accuracy_score,
    }.get(selection_metric)
    if metric_fn is None:
        raise ValueError("selection_metric must be 'accuracy' or 'balanced_accuracy'.")

    # Default compact grid
    if grid is None:
        grid = {
            "criterion": ["gini", "entropy", "log_loss"],
            "max_depth": [2, 3, 4, 5],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 5, 10],
            "ccp_alpha": [0.0, 1e-4, 1e-3, 1e-2],
        }

    # Base pipeline: impute then fit tree
    def make_model(params: Dict[str, Any]) -> Pipeline:
        return Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("clf", DecisionTreeClassifier(random_state=random_state, **params)),
            ]
        )

    # Iterate grid
    def _iter_grid(g: Dict[str, Any]):
        keys = list(g.keys())
        vals = [g[k] for k in keys]
        for combo in np.array(np.meshgrid(*vals, indexing="ij"), dtype=object).reshape(len(keys), -1).T:
            yield {k: v for k, v in zip(keys, combo)}

    best_model: Optional[Pipeline] = None
    best_params: Optional[Dict[str, Any]] = None
    best_val_score = -np.inf
    rows = []

    for params in _iter_grid(grid):
        model = make_model(params)
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        val_score = float(metric_fn(y_val, val_pred))

        rows.append({**params, f"val_{selection_metric}": val_score})

        if val_score > best_val_score:
            best_val_score = val_score
            best_model = model
            best_params = params

    assert best_model is not None and best_params is not None

    # Final evaluation on test
    val_pred = best_model.predict(X_val)
    test_pred = best_model.predict(X_test)

    report = {
        "label_col": label_col,
        "feature_cols": feature_cols,
        "selection_metric": selection_metric,
        "best_params": best_params,
        f"valid_{selection_metric}": float(metric_fn(y_val, val_pred)),
        "valid_accuracy": float(accuracy_score(y_val, val_pred)),
        "valid_balanced_accuracy": float(balanced_accuracy_score(y_val, val_pred)),
        "test_accuracy": float(accuracy_score(y_test, test_pred)),
        "test_balanced_accuracy": float(balanced_accuracy_score(y_test, test_pred)),
        "test_confusion_matrix": confusion_matrix(y_test, test_pred),
        "test_classification_report": classification_report(y_test, test_pred, digits=4),
    }

    leaderboard = None
    if return_leaderboard:
        leaderboard = (
            pd.DataFrame(rows)
            .sort_values(by=f"val_{selection_metric}", ascending=False)
            .reset_index(drop=True)
        )

    # Convenience: print a compact summary
    print(f"[{label_col}] best valid {selection_metric}: {best_val_score:.4f}")
    print(f"Best params: {best_params}")
    print(f"Test accuracy: {report['test_accuracy']:.4f} | Test bal-acc: {report['test_balanced_accuracy']:.4f}")
    print("\nTest classification report:\n", report["test_classification_report"])

    return best_model, report, leaderboard


In [ ]:
valid_df = pd.DataFrame.from_dict(valid_arc, orient="index")
test_df = pd.DataFrame.from_dict(test_arc, orient="index")
best_model, report, leaderboard = train_select_dt_and_test(
    train_augmented_df=train_augmented_arc_df,
    valid_df=valid_df,
    test_df=test_df,
    label_col="incision_architecture_rating",
    feature_cols=['min_ratio', 'max_ratio', 'mean_deviation', 'max_deviation', 'min_area', 'max_area', 'area_growth_slope_early', 'area_growth_slope_late', 'longest_detection_run'],
    selection_metric="accuracy"
)
leaderboard.head(10)

[incision_architecture_rating] best valid accuracy: 0.8667
Best params: {'criterion': 'gini', 'max_depth': 2, 'min_samples_split': 2, 'min_samples_leaf': 10, 'ccp_alpha': 0.0}
Test accuracy: 0.7333 | Test bal-acc: 0.5795

Test classification report:
               precision    recall  f1-score   support

         0.0     0.5000    0.2500    0.3333         4
         1.0     0.7692    0.9091    0.8333        11

    accuracy                         0.7333        15
   macro avg     0.6346    0.5795    0.5833        15
weighted avg     0.6974    0.7333    0.7000        15



,criterion,max_depth,min_samples_split,min_samples_leaf,ccp_alpha,val_accuracy
0,gini,2,5,10,0.0010,0.866667
1,gini,2,5,10,0.0001,0.866667
2,gini,2,5,10,0.0000,0.866667
3,gini,2,2,10,0.0100,0.866667
4,gini,2,2,10,0.0010,0.866667
5,gini,2,2,10,0.0001,0.866667
6,gini,2,2,10,0.0000,0.866667
7,gini,2,10,10,0.0000,0.866667
8,gini,2,10,10,0.0001,0.866667
9,gini,2,5,10,0.0100,0.866667


In [ ]:
best_model.named_steps

{'imputer': SimpleImputer(strategy='median'),
 'clf': DecisionTreeClassifier(max_depth=2, min_samples_leaf=10, random_state=42)}

In [ ]:
splits = extract_tree_splits(best_model.named_steps['clf'], ['min_ratio', 'max_ratio', 'mean_deviation', 'max_deviation', 'min_area', 'max_area', 'area_growth_slope_early', 'area_growth_slope_late', 'longest_detection_run'])
for s in splits:
    print(s)

{'node': 0, 'feature': 'longest_detection_run', 'threshold': np.float64(51.5), 'n_samples': np.int64(200)}
{'node': 1, 'feature': 'area_growth_slope_early', 'threshold': np.float64(-9.772916793823242), 'n_samples': np.int64(136)}
{'node': 4, 'feature': 'area_growth_slope_late', 'threshold': np.float64(0.5757575631141663), 'n_samples': np.int64(64)}


In [ ]:
pickle.dump(best_model, open(str(MODELS_DIR / "best_tree_arc.pkl"), "wb"))

In [ ]:
valid_df = pd.DataFrame.from_dict(valid_loc, orient="index")
test_df = pd.DataFrame.from_dict(test_loc, orient="index")
best_model, report, leaderboard = train_select_dt_and_test(
    train_augmented_df=train_augmented_loc_df,
    valid_df=valid_df,
    test_df=test_df,
    label_col="incision_location_rating",
    feature_cols=['min_ratio', 'max_ratio', 'mean_deviation', 'max_deviation', 'min_area', 'max_area', 'area_growth_slope_early', 'area_growth_slope_late', 'longest_detection_run'],
    selection_metric="accuracy"
)
leaderboard.head(10)

[incision_location_rating] best valid accuracy: 0.7222
Best params: {'criterion': 'gini', 'max_depth': 2, 'min_samples_split': 2, 'min_samples_leaf': 1, 'ccp_alpha': 0.0}
Test accuracy: 0.5263 | Test bal-acc: 0.2381

Test classification report:
               precision    recall  f1-score   support

        -1.0     0.0000    0.0000    0.0000         1
         0.0     0.0000    0.0000    0.0000         4
         1.0     0.7143    0.7143    0.7143        14

    accuracy                         0.5263        19
   macro avg     0.2381    0.2381    0.2381        19
weighted avg     0.5263    0.5263    0.5263        19



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,criterion,max_depth,min_samples_split,min_samples_leaf,ccp_alpha,val_accuracy
0,gini,5,20,10,0.0100,0.722222
1,gini,5,20,10,0.0010,0.722222
2,gini,5,20,10,0.0001,0.722222
3,gini,5,20,10,0.0000,0.722222
4,gini,5,20,5,0.0100,0.722222
5,gini,5,20,5,0.0010,0.722222
6,gini,5,20,5,0.0001,0.722222
7,gini,5,20,5,0.0000,0.722222
8,gini,5,20,2,0.0100,0.722222
9,gini,5,20,2,0.0010,0.722222


In [ ]:
#train_ratio=0.50, val_ratio=0.25, test_ratio=0.25, seed=42
valid_df = pd.DataFrame.from_dict(valid_size, orient="index")
test_df = pd.DataFrame.from_dict(test_size, orient="index")
best_model, report, leaderboard = train_select_dt_and_test(
    train_augmented_df=train_augmented_size_df,
    valid_df=valid_df,
    test_df=test_df,
    label_col="incision_size",
    feature_cols=['min_ratio', 'max_ratio', 'mean_deviation', 'max_deviation', 'min_area', 'max_area', 'area_growth_slope_early', 'area_growth_slope_late', 'longest_detection_run'],
    selection_metric="accuracy"
)
leaderboard.head(10)

[incision_size] best valid accuracy: 0.9333
Best params: {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1, 'ccp_alpha': 0.0}
Test accuracy: 0.8000 | Test bal-acc: 0.6250

Test classification report:
               precision    recall  f1-score   support

         0.0     1.0000    0.2500    0.4000         4
         1.0     0.7857    1.0000    0.8800        11

    accuracy                         0.8000        15
   macro avg     0.8929    0.6250    0.6400        15
weighted avg     0.8429    0.8000    0.7520        15



,criterion,max_depth,min_samples_split,min_samples_leaf,ccp_alpha,val_accuracy
0,gini,5,5,1,0.0010,0.933333
1,gini,5,5,1,0.0001,0.933333
2,gini,5,5,1,0.0000,0.933333
3,gini,5,10,1,0.0100,0.933333
4,gini,5,10,1,0.0010,0.933333
5,gini,5,10,1,0.0001,0.933333
6,gini,5,2,1,0.0000,0.933333
7,gini,5,2,1,0.0001,0.933333
8,gini,5,2,1,0.0010,0.933333
9,gini,5,2,1,0.0100,0.933333


In [ ]:
splits = extract_tree_splits(best_model.named_steps['clf'], ['min_ratio', 'max_ratio', 'mean_deviation', 'max_deviation', 'min_area', 'max_area', 'area_growth_slope_early', 'area_growth_slope_late', 'longest_detection_run'])
for s in splits:
    print(s)

{'node': 0, 'feature': 'longest_detection_run', 'threshold': np.float64(11.5), 'n_samples': np.int64(171)}
{'node': 1, 'feature': 'max_ratio', 'threshold': np.float64(1.5597155690193176), 'n_samples': np.int64(26)}
{'node': 4, 'feature': 'min_ratio', 'threshold': np.float64(0.47874823212623596), 'n_samples': np.int64(145)}
{'node': 5, 'feature': 'max_area', 'threshold': np.float64(944.0), 'n_samples': np.int64(60)}
{'node': 6, 'feature': 'longest_detection_run', 'threshold': np.float64(29.5), 'n_samples': np.int64(48)}
{'node': 8, 'feature': 'min_ratio', 'threshold': np.float64(0.3339662104845047), 'n_samples': np.int64(43)}
{'node': 12, 'feature': 'longest_detection_run', 'threshold': np.float64(15.5), 'n_samples': np.int64(85)}
{'node': 13, 'feature': 'max_ratio', 'threshold': np.float64(1.3907347917556763), 'n_samples': np.int64(11)}
{'node': 16, 'feature': 'min_ratio', 'threshold': np.float64(0.7555866837501526), 'n_samples': np.int64(74)}
{'node': 17, 'feature': 'max_area', 'thres

In [ ]:
pickle.dump(best_model, open(str(MODELS_DIR / "best_tree_size.pkl"), "wb"))

In [ ]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple, List
from itertools import product

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def train_select_mlp_and_test(
    train_augmented_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: List[str],
    label_col: str,
    *,
    selection_metric: str = "balanced_accuracy",
    grid: Optional[Dict[str, Any]] = None,
    random_state: int = 42,
    return_leaderboard: bool = True,
    labels: Optional[List[int]] = None,  # e.g. [-1, 0, 1] for stable reporting
) -> Tuple[Pipeline, Dict[str, Any], Optional[pd.DataFrame]]:
    # --- checks ---
    for df_name, df in [("train_augmented_df", train_augmented_df), ("valid_df", valid_df), ("test_df", test_df)]:
        if label_col not in df.columns:
            raise ValueError(f"{label_col=} not in {df_name} columns")

    missing_feats = [c for c in feature_cols if c not in train_augmented_df.columns]
    if missing_feats:
        raise ValueError(f"Missing feature columns in train_augmented_df: {missing_feats}")

    def _Xy(df: pd.DataFrame):
        X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
        y = df[label_col].to_numpy()
        return X, y

    X_train, y_train = _Xy(train_augmented_df)
    X_val, y_val = _Xy(valid_df)
    X_test, y_test = _Xy(test_df)

    metric_fn = {
        "accuracy": accuracy_score,
        "balanced_accuracy": balanced_accuracy_score,
    }.get(selection_metric)
    if metric_fn is None:
        raise ValueError("selection_metric must be 'accuracy' or 'balanced_accuracy'.")

    # --- default grid (safe for tuples like hidden_layer_sizes) ---
    if grid is None:
        grid = {
            "hidden_layer_sizes": [(16,), (32,), (64,), (32, 16), (64, 32)],
            "activation": ["relu", "tanh"],
            "alpha": [1e-5, 1e-4, 1e-3, 1e-2],
            "learning_rate_init": [1e-3, 3e-4, 1e-4],
            "batch_size": [32, 64, 128],
            "max_iter": [500],
            "early_stopping": [True],
            "n_iter_no_change": [20],
        }

    def _iter_grid(g: Dict[str, Any]):
        keys = list(g.keys())
        values = [g[k] if isinstance(g[k], (list, tuple)) else [g[k]] for k in keys]
        for combo in product(*values):
            yield dict(zip(keys, combo))

    def make_model(params: Dict[str, Any]) -> Pipeline:
        return Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("clf", MLPClassifier(random_state=random_state, solver="adam", **params)),
            ]
        )

    best_model: Optional[Pipeline] = None
    best_params: Optional[Dict[str, Any]] = None
    best_val_score = -np.inf
    rows = []

    for params in _iter_grid(grid):
        model = make_model(params)
        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)
        val_score = float(metric_fn(y_val, val_pred))
        rows.append({**params, f"val_{selection_metric}": val_score})

        if val_score > best_val_score:
            best_val_score = val_score
            best_model = model
            best_params = params

    assert best_model is not None and best_params is not None

    # --- evaluate ---
    val_pred = best_model.predict(X_val)
    test_pred = best_model.predict(X_test)

    cm = confusion_matrix(y_test, test_pred, labels=labels) if labels is not None else confusion_matrix(y_test, test_pred)
    cr = classification_report(
        y_test, test_pred, labels=labels, digits=4, zero_division=0
    ) if labels is not None else classification_report(y_test, test_pred, digits=4, zero_division=0)

    report = {
        "label_col": label_col,
        "feature_cols": feature_cols,
        "selection_metric": selection_metric,
        "best_params": best_params,
        f"valid_{selection_metric}": float(metric_fn(y_val, val_pred)),
        "valid_accuracy": float(accuracy_score(y_val, val_pred)),
        "valid_balanced_accuracy": float(balanced_accuracy_score(y_val, val_pred)),
        "test_accuracy": float(accuracy_score(y_test, test_pred)),
        "test_balanced_accuracy": float(balanced_accuracy_score(y_test, test_pred)),
        "test_confusion_matrix": cm,
        "test_classification_report": cr,
    }

    leaderboard = None
    if return_leaderboard:
        leaderboard = (
            pd.DataFrame(rows)
            .sort_values(by=f"val_{selection_metric}", ascending=False)
            .reset_index(drop=True)
        )

    print(f"[{label_col}] best valid {selection_metric}: {best_val_score:.4f}")
    print(f"Best params: {best_params}")
    print(f"Test accuracy: {report['test_accuracy']:.4f} | Test bal-acc: {report['test_balanced_accuracy']:.4f}")
    print("\nTest classification report:\n", report["test_classification_report"])

    return best_model, report, leaderboard


In [ ]:
import pickle

In [ ]:
valid_arc_df = pd.DataFrame.from_dict(valid_arc, orient="index")
test_arc_df = pd.DataFrame.from_dict(test_arc, orient="index")
best_mlp, rep, lb = train_select_mlp_and_test(
    train_augmented_arc_df, valid_arc_df, test_arc_df,
    feature_cols=['min_ratio', 'max_ratio', 'mean_deviation', 'max_deviation', 'min_area', 'max_area', 'area_growth_slope_early', 'area_growth_slope_late', 'longest_detection_run'],
    label_col="incision_architecture_rating",
    selection_metric="accuracy",
    labels=[-1, 0, 1],
)
lb.head(10)

[incision_architecture_rating] best valid accuracy: 0.8000
Best params: {'hidden_layer_sizes': (32,), 'activation': 'relu', 'alpha': 1e-05, 'learning_rate_init': 0.0003, 'batch_size': 128, 'max_iter': 500, 'early_stopping': True, 'n_iter_no_change': 20}
Test accuracy: 0.7333 | Test bal-acc: 0.5795

Test classification report:
               precision    recall  f1-score   support

          -1     0.0000    0.0000    0.0000         0
           0     0.5000    0.2500    0.3333         4
           1     0.7692    0.9091    0.8333        11

    accuracy                         0.7333        15
   macro avg     0.4231    0.3864    0.3889        15
weighted avg     0.6974    0.7333    0.7000        15



,hidden_layer_sizes,activation,alpha,learning_rate_init,batch_size,max_iter,early_stopping,n_iter_no_change,val_accuracy
0,"(32,)",relu,0.01000,0.0001,32,500,True,20,0.8
1,"(32,)",relu,0.01000,0.0003,128,500,True,20,0.8
2,"(32,)",relu,0.01000,0.0001,128,500,True,20,0.8
3,"(32,)",relu,0.01000,0.0001,64,500,True,20,0.8
4,"(32,)",relu,0.00010,0.0001,32,500,True,20,0.8
5,"(32,)",relu,0.00010,0.0003,128,500,True,20,0.8
6,"(32,)",relu,0.00010,0.0001,128,500,True,20,0.8
7,"(32,)",relu,0.00010,0.0001,64,500,True,20,0.8
8,"(32,)",relu,0.00001,0.0001,128,500,True,20,0.8
9,"(32,)",relu,0.00001,0.0001,64,500,True,20,0.8


In [ ]:
pickle.dump(best_mlp, open(str(MODELS_DIR / "best_mlp_arc.pkl"), "wb"))

In [ ]:
valid_loc_df = pd.DataFrame.from_dict(valid_loc, orient="index")
test_loc_df = pd.DataFrame.from_dict(test_loc, orient="index")
best_mlp, rep, lb = train_select_mlp_and_test(
    train_augmented_loc_df, valid_loc_df, test_loc_df,
    feature_cols=['min_ratio', 'max_ratio', 'mean_deviation', 'max_deviation', 'min_area', 'max_area', 'area_growth_slope_early', 'area_growth_slope_late', 'longest_detection_run'],
    label_col="incision_location_rating",
    selection_metric="accuracy",
    labels=[-1, 0, 1],
)
lb.head(10)

[incision_location_rating] best valid accuracy: 0.7222
Best params: {'hidden_layer_sizes': (32, 16), 'activation': 'relu', 'alpha': 1e-05, 'learning_rate_init': 0.001, 'batch_size': 32, 'max_iter': 500, 'early_stopping': True, 'n_iter_no_change': 20}
Test accuracy: 0.5263 | Test bal-acc: 0.2381

Test classification report:
               precision    recall  f1-score   support

          -1     0.0000    0.0000    0.0000         1
           0     0.0000    0.0000    0.0000         4
           1     0.6667    0.7143    0.6897        14

    accuracy                         0.5263        19
   macro avg     0.2222    0.2381    0.2299        19
weighted avg     0.4912    0.5263    0.5082        19



,hidden_layer_sizes,activation,alpha,learning_rate_init,batch_size,max_iter,early_stopping,n_iter_no_change,val_accuracy
0,"(64, 32)",relu,0.01000,0.0003,64,500,True,20,0.722222
1,"(64, 32)",relu,0.01000,0.0003,32,500,True,20,0.722222
2,"(64, 32)",relu,0.01000,0.0010,128,500,True,20,0.722222
3,"(64, 32)",relu,0.01000,0.0010,64,500,True,20,0.722222
4,"(64, 32)",relu,0.00100,0.0003,64,500,True,20,0.722222
5,"(64, 32)",relu,0.00100,0.0003,32,500,True,20,0.722222
6,"(64, 32)",relu,0.00100,0.0010,128,500,True,20,0.722222
7,"(64, 32)",relu,0.00100,0.0010,64,500,True,20,0.722222
8,"(32, 16)",relu,0.00001,0.0010,32,500,True,20,0.722222
9,"(64, 32)",relu,0.00010,0.0010,64,500,True,20,0.722222


In [ ]:
valid_size_df = pd.DataFrame.from_dict(valid_size, orient="index")
test_size_df = pd.DataFrame.from_dict(test_size, orient="index")
best_mlp, rep, lb = train_select_mlp_and_test(
    train_augmented_size_df, valid_size_df, test_size_df,
    feature_cols=['min_ratio', 'max_ratio', 'mean_deviation', 'max_deviation', 'min_area', 'max_area', 'area_growth_slope_early', 'area_growth_slope_late', 'longest_detection_run'],
    label_col="incision_size",
    selection_metric="accuracy",
    labels=[-1, 0, 1],
)
lb.head(10)

[incision_size] best valid accuracy: 0.9333
Best params: {'hidden_layer_sizes': (64,), 'activation': 'relu', 'alpha': 1e-05, 'learning_rate_init': 0.0001, 'batch_size': 64, 'max_iter': 500, 'early_stopping': True, 'n_iter_no_change': 20}
Test accuracy: 0.6667 | Test bal-acc: 0.4545

Test classification report:
               precision    recall  f1-score   support

          -1     0.0000    0.0000    0.0000         0
           0     0.0000    0.0000    0.0000         4
           1     0.7143    0.9091    0.8000        11

    accuracy                         0.6667        15
   macro avg     0.2381    0.3030    0.2667        15
weighted avg     0.5238    0.6667    0.5867        15



,hidden_layer_sizes,activation,alpha,learning_rate_init,batch_size,max_iter,early_stopping,n_iter_no_change,val_accuracy
0,"(64,)",relu,0.00100,0.0001,64,500,True,20,0.933333
1,"(64,)",relu,0.00010,0.0001,64,500,True,20,0.933333
2,"(64,)",relu,0.00001,0.0001,64,500,True,20,0.933333
3,"(64,)",relu,0.01000,0.0001,64,500,True,20,0.933333
4,"(64,)",relu,0.00001,0.0003,32,500,True,20,0.866667
5,"(64,)",relu,0.00001,0.0003,64,500,True,20,0.866667
6,"(64,)",relu,0.00010,0.0003,32,500,True,20,0.866667
7,"(64,)",relu,0.00100,0.0003,128,500,True,20,0.866667
8,"(64,)",relu,0.00100,0.0001,32,500,True,20,0.866667
9,"(64,)",relu,0.00100,0.0003,32,500,True,20,0.866667


In [ ]:
pickle.dump(best_mlp, open(str(MODELS_DIR / "best_mlp_size.pkl"), "wb"))

In [ ]:
# combine -1 and 0 before augmentation
# incorporate the triangle phase, separate area/ratio for the two phases, time(number of frames) within the two phases
# create comparison:
#     qualitative comparisons for the incision wound in a figure (frame of videa, our seg results, SAM/SAM2 point prompt, SAM/SAM2 bounding box, MED SAM point pronpt) three good videos
#     feature of case videos -> build tree/mlp see prediciton results
